# Stock Price Time-Series Analysis

This notebook downloads historical Apple (AAPL) prices from a public GitHub CSV. It explores the data, parses dates, plots price trends and moving averages, checks monthly/weekly patterns and volume relationships, and makes a simple exponential-smoothing forecast.

This is an educational time-series exercise, not investment advice.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
URL = 'https://raw.githubusercontent.com/plotly/datasets/master/finance-charts-apple.csv'
df = pd.read_csv(URL)
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.head())
display(df.dtypes.to_frame('data_type'))

## Parse dates and inspect prices

In [ ]:
# The public dataset calls its date field 'Date' and its closing price 'AAPL.Close'.
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date']).sort_values('Date').drop_duplicates(subset='Date').set_index('Date')
PRICE_COL = 'AAPL.Close'
VOLUME_COL = 'AAPL.Volume'
print('Period:', df.index.min().date(), 'to', df.index.max().date())
display(df[[PRICE_COL, VOLUME_COL]].describe().T)

## Historical price trend and moving averages

In [ ]:
ax = df[PRICE_COL].plot(figsize=(12,5), label='Daily closing price', color='steelblue', linewidth=1)
df[PRICE_COL].rolling(20).mean().plot(ax=ax, label='20-day moving average', color='orange', linewidth=1.5)
df[PRICE_COL].rolling(50).mean().plot(ax=ax, label='50-day moving average', color='green', linewidth=1.5)
ax.set_title('AAPL closing price and moving averages')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend()
plt.tight_layout()
plt.show()

## Weekly, monthly, and yearly patterns

Compare average closing prices by calendar period. These summaries show patterns in this dataset and do not guarantee recurring future behavior.

In [ ]:
df['DailyReturnPct'] = df[PRICE_COL].pct_change() * 100
monthly = df[PRICE_COL].resample('MS').mean()
weekly = df[PRICE_COL].resample('W').mean()
yearly = df[PRICE_COL].resample('YS').mean()
yearly.index = yearly.index.year
fig, axes = plt.subplots(3,1,figsize=(12,11))
monthly.plot(ax=axes[0], color='purple', title='Monthly average closing price')
weekly.plot(ax=axes[1], color='teal', title='Weekly average closing price')
yearly.plot(kind='bar', ax=axes[2], color='slateblue', title='Yearly average closing price')
for ax in axes:
    ax.set_ylabel('Price (USD)')
plt.tight_layout()
plt.show()

## Price and trading volume relationship

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
df[PRICE_COL].plot(ax=axes[0], color='steelblue', title='Closing price')
df[VOLUME_COL].plot(ax=axes[1], color='darkorange', title='Trading volume')
for ax in axes:
    ax.set_xlabel('Date')
plt.tight_layout()
plt.show()
print('Correlation between price and volume:', round(df[PRICE_COL].corr(df[VOLUME_COL]), 3))
plt.figure(figsize=(6,4))
sns.scatterplot(data=df, x=VOLUME_COL, y=PRICE_COL, alpha=0.5)
plt.title('Closing price vs trading volume')
plt.tight_layout()
plt.show()

## Simple future forecast with exponential smoothing

Forecast the next 30 business days from the closing-price series. This basic model does not account for market news or external drivers. Install `statsmodels` if your environment does not have it (`%pip install statsmodels`).

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

series = df[PRICE_COL].asfreq('B').ffill()
model = ExponentialSmoothing(series, trend='add', seasonal=None, initialization_method='estimated').fit()
forecast = model.forecast(30)
plt.figure(figsize=(12,5))
series.tail(180).plot(label='Recent closing prices')
forecast.plot(label='30-business-day forecast', color='red')
plt.title('Simple exponential-smoothing forecast')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.show()
display(forecast.to_frame('forecast_price'))